In [1]:
import logging

logging.basicConfig(level=logging.DEBUG)

from pathlib import Path
from toric_spines_sim.geometry.sink import SinkGeometry, append_sink_to_swc
from swctools import SWCModel, plot_model, PointSet, FrustaSet

DEBUG:matplotlib:matplotlib data path: /home/jordan/repos/toric_spines_sim/.venv/lib/python3.12/site-packages/matplotlib/mpl-data
DEBUG:matplotlib:CONFIGDIR=/home/jordan/.config/matplotlib
DEBUG:matplotlib:interactive is False
DEBUG:matplotlib:platform is linux
DEBUG:matplotlib:CACHEDIR=/home/jordan/.cache/matplotlib
DEBUG:matplotlib.font_manager:Using fontManager instance from /home/jordan/.cache/matplotlib/fontlist-v390.json
INFO:numexpr.utils:NumExpr defaulting to 8 threads.


In [2]:
um_per_px = 5 / 1000  # microns per pixel
sink_radius_um = 20
sink_connector_length_um = 5

sink_radius_px = sink_radius_um / um_per_px
sink_connector_length_px = sink_connector_length_um / um_per_px

swc_spine_px_filepath = Path("../data/swc/pixels/TS3_s200_equivalent_area.swc")
neckpt_px_filepath = Path("../data/pointsets/pixels/TS3_neckpoint.txt")
swc_with_sink_px_filepath = Path(
    f"../data/swc/pixels/TS3_s200_wsink_r{sink_radius_um}um.swc"
)
swc_with_sink_um_filepath = Path(
    f"../data/swc/microns/TS3_s200_wsink_r{sink_radius_um}um.swc"
)
az_px_filepath = Path("../data/pointsets/pixels/TS3_AZ.txt")
synpts_um_filepath = Path("../data/pointsets/microns/TS3_synpts.txt")

swc_spine_px = SWCModel.from_swc_file(swc_spine_px_filepath)
spine_frusta_px = FrustaSet.from_swc_model(swc_spine_px, sides=20, end_caps=False)

az_pointset_px = PointSet.from_txt_file(az_px_filepath)
az_pointset_px_proj = az_pointset_px.project_onto_frusta(spine_frusta_px)
synpts_pointset_um = az_pointset_px_proj.scale(um_per_px)
synpts_pointset_um.to_txt_file(synpts_um_filepath)

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
DEBUG:swctools.io:reconnect header i=60 j=2 line=5
DEBUG:swctools.io:reconnect header i=61 j=11 line=6
DEBUG:swctools.io:reconnect header i=62 j=28 line=7
DEBUG:swctools.io:reconnect header i=63 j=35 line=8
DEBUG:swctools.io:reconnect header i=64 j=35 line=9
DEBUG:swctools.io:reconnect header i=65 j=38 line=10
INFO:swctools.io:parse_swc done records=65 reconnections=6 header=10
INFO:swctools.model:SWCModel.from_parse_result records=65 reconnections=6 header=10
INFO:swctools.model:SWCModel.from_swc_file built nodes=65 edges=64 strict=True validate_reconnections=True
DEBUG:swctools.geometry:frustum_mesh sides=20 end_caps=False verts=40 faces=40
DEBUG:swctools.geometry:frustum_mesh sides=20 end_caps=False verts=40 faces=40
DEBUG:swctools.geometry:frustum_mesh sides=20 end_caps=False verts=40 faces=40
DEBUG:swctools.geometry:frustum_mesh sides=20 end_caps=False verts=40 faces=40
DEBUG:swctools.geometry

In [3]:
swc_spine_model_px = SWCModel.from_swc_file(swc_spine_px_filepath)
neckpt_pointset_px = PointSet.from_txt_file(neckpt_px_filepath)
plot_model(swc_model=swc_spine_model_px, point_set=neckpt_pointset_px, point_size=10)

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
DEBUG:swctools.io:reconnect header i=60 j=2 line=5
DEBUG:swctools.io:reconnect header i=61 j=11 line=6
DEBUG:swctools.io:reconnect header i=62 j=28 line=7
DEBUG:swctools.io:reconnect header i=63 j=35 line=8
DEBUG:swctools.io:reconnect header i=64 j=35 line=9
DEBUG:swctools.io:reconnect header i=65 j=38 line=10
INFO:swctools.io:parse_swc done records=65 reconnections=6 header=10
INFO:swctools.model:SWCModel.from_parse_result records=65 reconnections=6 header=10
INFO:swctools.model:SWCModel.from_swc_file built nodes=65 edges=64 strict=True validate_reconnections=True
INFO:swctools.geometry:PointSet.from_txt_file path=../data/pointsets/pixels/TS3_neckpoint.txt n=3
DEBUG:swctools.geometry:sphere_mesh stacks=6 slices=12 verts=62 faces=120
DEBUG:swctools.geometry:sphere_mesh stacks=6 slices=12 verts=62 faces=120
DEBUG:swctools.geometry:sphere_mesh stacks=6 slices=12 verts=62 faces=120
INFO:swctools.geome

In [4]:
from toric_spines_sim.geometry.sink import append_sink_to_swc_multi_neck_points

# add sink in px and write swc out
geom = SinkGeometry(
    radius=sink_radius_px,
    length=2 * sink_radius_px,
    n_segments=5,
    axis="-z",
    connector_length=sink_connector_length_px,
)  # radius 1000 px = 5 um
out_path = append_sink_to_swc_multi_neck_points(
    swc_in=swc_spine_px_filepath,
    swc_out=swc_with_sink_px_filepath,
    neck_points=neckpt_px_filepath,  # or (x,y,z)
    geom=geom,
    tag=5,
)

# convert to um and write swc out
swc_model_with_sink_px = SWCModel.from_swc_file(swc_with_sink_px_filepath)
swc_model_with_sink_um = swc_model_with_sink_px.scale(um_per_px)
swc_model_with_sink_um.to_swc_file(swc_with_sink_um_filepath)

DEBUG:toric_spines_sim.model:Read 65 SWC points from ../data/swc/pixels/TS3_s200_equivalent_area.swc
DEBUG:toric_spines_sim.sink:snapped primary neck point to node 15 at (6415.519984, 6437.861242, 3091.229237)
DEBUG:toric_spines_sim.sink:generated 6 sink points
DEBUG:toric_spines_sim.sink:wrote 8 lines to ../data/swc/pixels/TS3_s200_wsink_r20um.swc
INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
DEBUG:swctools.io:reconnect header i=60 j=2 line=5
DEBUG:swctools.io:reconnect header i=61 j=11 line=6
DEBUG:swctools.io:reconnect header i=62 j=28 line=7
DEBUG:swctools.io:reconnect header i=63 j=35 line=8
DEBUG:swctools.io:reconnect header i=64 j=35 line=9
DEBUG:swctools.io:reconnect header i=65 j=38 line=10
INFO:swctools.io:parse_swc done records=73 reconnections=6 header=13
INFO:swctools.model:SWCModel.from_parse_result records=73 reconnections=6 header=13
INFO:swctools.model:SWCModel.from_swc_file built nodes=73 edges=72 strict=True validate_reconne

In [5]:
fig = plot_model(
    swc_model=swc_model_with_sink_um,
    point_set=synpts_pointset_um,
    point_size=10,
    point_color="crimson",
)

# fig.update_layout(
#     scene=dict(
#         xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False)
#     )
# )

fig.show()

DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_ca